# 01 · Preparación del dataset — Rice Image Dataset

**Tema 1 del proyecto.** Este notebook deja lista la base que usan todos los demás:

1. Inventario de las imágenes y control de integridad.
2. Eliminación de duplicados exactos (prevención de *data leakage*).
3. Partición estratificada **70 % train / 15 % validación / 15 % test**.
4. Subconjunto estratificado para Hyperband y validación cruzada.
5. Reproducibilidad: todas las distribuciones quedan guardadas en archivos CSV.
6. Preprocesamiento y **generador de datos** que alimenta el modelo lote por lote.

No necesita GPU. Entrada: el dataset `muratkokluu/rice-image-dataset` adjunto al notebook.
Salida (en `/kaggle/working`): `train.csv`, `val.csv`, `test.csv`, `subset_train.csv`,
`subset_val.csv`, `resumen_particion.csv` y `rice_data.py`.

In [ ]:
import hashlib
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

## 1. Módulo compartido `rice_data.py`

Las constantes y el generador se escriben en un archivo `.py` para que los notebooks
siguientes importen exactamente el mismo código (una sola fuente, sin copiar y pegar).

In [ ]:
%%writefile rice_data.py
"""Utilidades compartidas del proyecto Rice Image Dataset (semilla, clases y generador)."""
import os
from pathlib import Path

import tensorflow as tf

SEED = 42
CLASSES = ["Arborio", "Basmati", "Ipsala", "Jasmine", "Karacadag"]
IMG_SIZE = 224


def find_data_root(base="/kaggle/input"):
    """Devuelve la carpeta que contiene las 5 subcarpetas de clases."""
    for root, dirs, _ in os.walk(base):
        if set(CLASSES) <= set(dirs):
            return Path(root)
    raise FileNotFoundError(f"No se encontraron las carpetas {CLASSES} dentro de {base}")


_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.1, fill_mode="constant", fill_value=0.0),
])


def _load(path, label, img_size):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (img_size, img_size))
    return img, label


def make_dataset(df, data_root, batch_size=32, training=False, img_size=IMG_SIZE, seed=SEED):
    """Generador por lotes a partir de un DataFrame con columnas filepath y label_idx.

    Lee las imágenes del disco solo cuando se necesitan, así el dataset nunca se carga
    completo en memoria. Entrega píxeles float32 en [0, 255]; la normalización propia de
    cada arquitectura va dentro del modelo. El aumento de datos solo se aplica si
    training=True.
    """
    paths = (str(data_root) + "/" + df["filepath"]).to_numpy()
    labels = df["label_idx"].to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(df), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: _load(p, y, img_size), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    if training:
        ds = ds.map(lambda x, y: (_augment(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
from rice_data import CLASSES, IMG_SIZE, SEED, find_data_root, make_dataset

DATA_ROOT = find_data_root()
print("Raíz del dataset:", DATA_ROOT)

## 2. Inventario

Las rutas se guardan **relativas** a la raíz del dataset (`Arborio/Arborio (1).jpg`), de modo
que los CSV sirven igual en Kaggle, en Colab o en un PC local.

In [ ]:
rows = []
for idx, cls in enumerate(CLASSES):
    for name in sorted(os.listdir(DATA_ROOT / cls)):
        if name.lower().endswith((".jpg", ".jpeg", ".png")):
            rows.append((f"{cls}/{name}", cls, idx))

df = pd.DataFrame(rows, columns=["filepath", "label", "label_idx"])
print("Total de imágenes:", len(df))
df["label"].value_counts()

In [ ]:
# Control de integridad sobre una muestra: tamaño y modo de color.
muestra = df.sample(300, random_state=SEED)
info = [Image.open(DATA_ROOT / p) for p in muestra["filepath"]]
print("Tamaños encontrados:", {im.size for im in info})
print("Modos de color:", {im.mode for im in info})

## 3. Duplicados exactos (*data leakage*)

Si la misma imagen cae en train y en test, el modelo es evaluado con algo que ya vio y la
métrica queda inflada. Se calcula el hash MD5 de cada archivo **antes** de partir:

- Duplicados con la misma etiqueta: se conserva una sola copia.
- Duplicados con etiquetas distintas: se eliminan todas las copias (etiqueta no confiable).

Limitación: el hash solo detecta copias idénticas byte a byte, no imágenes casi iguales.

In [ ]:
def md5(rel_path):
    return hashlib.md5((DATA_ROOT / rel_path).read_bytes()).hexdigest()

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    df["md5"] = list(ex.map(md5, df["filepath"]))
print(f"Hash calculado en {time.time() - t0:.0f} s")

etiquetas_por_hash = df.groupby("md5")["label"].nunique()
conflictivos = etiquetas_por_hash[etiquetas_por_hash > 1].index
n_inicial = len(df)
df = df[~df["md5"].isin(conflictivos)]
df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)

print("Hashes con etiquetas en conflicto:", len(conflictivos))
print("Imágenes eliminadas:", n_inicial - len(df))
print("Imágenes restantes:", len(df))

## 4. Partición estratificada 70 / 15 / 15

Estratificar mantiene la misma proporción de clases en los tres conjuntos. La semilla fija
(`SEED = 42`) hace que la partición sea idéntica en cada ejecución.

- **train**: ajuste de pesos.
- **val**: selección de hiperparámetros y *early stopping*.
- **test**: se usa una sola vez, al final, para reportar el desempeño.

In [ ]:
train_df, resto = train_test_split(df, test_size=0.30, stratify=df["label_idx"], random_state=SEED)
val_df, test_df = train_test_split(resto, test_size=0.50, stratify=resto["label_idx"], random_state=SEED)

particiones = {"train": train_df, "val": val_df, "test": test_df}

# Verificación: ningún archivo ni hash aparece en más de un conjunto.
nombres = list(particiones)
for i in range(3):
    for j in range(i + 1, 3):
        a, b = particiones[nombres[i]], particiones[nombres[j]]
        assert not set(a["filepath"]) & set(b["filepath"]), "archivo repetido entre conjuntos"
        assert not set(a["md5"]) & set(b["md5"]), "imagen duplicada entre conjuntos"
assert sum(len(p) for p in particiones.values()) == len(df)

resumen = pd.DataFrame({n: p["label"].value_counts().sort_index() for n, p in particiones.items()})
resumen.loc["TOTAL"] = resumen.sum()
resumen.loc["PROPORCIÓN"] = (resumen.loc["TOTAL"] / len(df)).round(4)
resumen

## 5. Subconjunto para Hyperband y validación cruzada

Buscar hiperparámetros y hacer k-fold de 5 modelos sobre todo el dataset no cabe en la cuota
de GPU. Se toma un subconjunto estratificado **solo de train** (2.000 por clase) con una
columna `fold` para el k-fold estratificado (k = 5), y otro **solo de val** (500 por clase).
Test no se toca.

In [ ]:
N_SUB_TRAIN, N_SUB_VAL, K = 2000, 500, 5

subset_train = (train_df.groupby("label_idx", group_keys=False)
                .sample(n=N_SUB_TRAIN, random_state=SEED).reset_index(drop=True))
subset_val = (val_df.groupby("label_idx", group_keys=False)
              .sample(n=N_SUB_VAL, random_state=SEED).reset_index(drop=True))

subset_train["fold"] = -1
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
for fold, (_, idx_val) in enumerate(skf.split(subset_train, subset_train["label_idx"])):
    subset_train.loc[idx_val, "fold"] = fold

pd.crosstab(subset_train["fold"], subset_train["label"])

## 6. Guardar las distribuciones en CSV

In [ ]:
columnas = ["filepath", "label", "label_idx", "md5"]
train_df[columnas].to_csv(OUT_DIR / "train.csv", index=False)
val_df[columnas].to_csv(OUT_DIR / "val.csv", index=False)
test_df[columnas].to_csv(OUT_DIR / "test.csv", index=False)
subset_train[columnas + ["fold"]].to_csv(OUT_DIR / "subset_train.csv", index=False)
subset_val[columnas].to_csv(OUT_DIR / "subset_val.csv", index=False)
resumen.to_csv(OUT_DIR / "resumen_particion.csv")
sorted(p.name for p in OUT_DIR.glob("*.csv"))

## 7. Preprocesamiento y generador de datos

`make_dataset` construye un `tf.data.Dataset` que lee, decodifica y redimensiona las imágenes
a 224×224 **lote por lote**, sin cargar el dataset completo en memoria.

- El aumento de datos (volteos y rotación leve) se aplica solo a train.
- La normalización depende de la arquitectura, así que va dentro de cada modelo; el generador
  entrega píxeles en [0, 255].

In [ ]:
train_ds = make_dataset(pd.read_csv(OUT_DIR / "train.csv"), DATA_ROOT, batch_size=32, training=True)
val_ds = make_dataset(pd.read_csv(OUT_DIR / "val.csv"), DATA_ROOT, batch_size=32)

x, y = next(iter(train_ds))
print("Lote de imágenes:", x.shape, x.dtype, "| rango:", float(x.numpy().min()), "-", float(x.numpy().max()))
print("Lote de etiquetas:", y.shape, y.numpy()[:10])

fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for ax, img, lab in zip(axes.ravel(), x, y):
    ax.imshow(img.numpy().astype("uint8"))
    ax.set_title(CLASSES[int(lab)])
    ax.axis("off")
plt.suptitle("Lote de entrenamiento (con aumento de datos)")
plt.show()

In [ ]:
# Velocidad del generador: imágenes por segundo en 50 lotes.
t0 = time.time()
n = sum(int(xb.shape[0]) for xb, _ in val_ds.take(50))
print(f"{n / (time.time() - t0):.0f} imágenes/s")

## Cómo se usa en los notebooks siguientes

1. En el notebook nuevo: **Add Input → Notebook Output →** este notebook.
2. Cargar el módulo y los CSV:

```python
import sys
sys.path.append("/kaggle/input/<nombre-de-este-notebook>")
import pandas as pd
from rice_data import make_dataset, find_data_root, CLASSES

SPLITS = "/kaggle/input/<nombre-de-este-notebook>"
train_ds = make_dataset(pd.read_csv(f"{SPLITS}/train.csv"), find_data_root(), training=True)
```